# Pré-processamento dos dados

## Objetivo
Preparar variáveis e dividir os dados para que o treinamento use somente informações disponíveis antes da ligação. A lógica reutilizável fica em `src/preprocessing.py`; este notebook importa as funções, sem duplicar o pré-processamento.

## Decisões principais
- A variável `duration` é removida por data leakage: só é conhecida após a ligação.
- `pdays=999` não significa 999 dias. Criamos `was_previously_contacted` e substituímos esse sentinela por ausente, a ser imputado pela mediana aprendida no treino.
- `unknown` é preservado como categoria própria.
- A divisão padrão é 80% treino e 20% teste, estratificada pelo alvo, com `random_state=42`.
- Imputação, padronização e categorias do One-Hot Encoder são aprendidas com treino; depois aplicadas ao teste.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import (
    PREVIOUS_CONTACT_FLAG,
    load_dataset,
    make_preprocessor,
    prepare_features_target,
    split_data,
)

print(f"Raiz do projeto: {PROJECT_ROOT}")

Raiz do projeto: C:\Users\User\Documents\Codex\2026-11-09\files-pasted-by-the-user-projeto


## Carregar os dados

`load_dataset()` lê o CSV oficial usando o separador `;`. Não fazemos transformação estatística nesta leitura; as transformações são preparadas depois da separação treino/teste.

In [2]:
df = load_dataset()
print(f"Dimensões do conjunto original: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
display(df.head())

Dimensões do conjunto original: 41,188 linhas × 21 colunas


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## Separar preditores e alvo; tratar `duration` e `pdays`

`prepare_features_target()` devolve `X` (preditores) e `y` (alvo). A função retira `duration` e `y` de `X`. Para `pdays`, marca os valores diferentes de `999` como clientes previamente contatados e converte o sentinela em ausente. O `SimpleImputer` tratará esses valores mais adiante, aprendendo a mediana apenas no treino.

In [3]:
X, y = prepare_features_target(df)
print(f"X: {X.shape[0]:,} linhas × {X.shape[1]} preditores")
print(f"y: {y.shape[0]:,} rótulos; categorias: {sorted(y.unique())}")
print(f"'duration' está em X? {'duration' in X.columns}")
print(f"Nulos em pdays depois de retirar o sentinela: {int(X['pdays'].isna().sum()):,}")
print("Indicador de contato anterior:")
display(X[PREVIOUS_CONTACT_FLAG].value_counts().sort_index().rename(index={0: "não contatado antes", 1: "contatado antes"}).to_frame("registros"))
print("Valores 'unknown' mantidos como categoria:")
unknown_counts = X.select_dtypes(exclude="number").eq("unknown").sum()
display(unknown_counts[unknown_counts.gt(0)].sort_values(ascending=False).to_frame("unknown"))

X: 41,188 linhas × 20 preditores
y: 41,188 rótulos; categorias: ['no', 'yes']
'duration' está em X? False
Nulos em pdays depois de retirar o sentinela: 39,673
Indicador de contato anterior:


,registros
was_previously_contacted,
não contatado antes,39673
contatado antes,1515


Valores 'unknown' mantidos como categoria:


,unknown
default,8597
education,1731
housing,990
loan,990
job,330
marital,80


A flag `was_previously_contacted` conserva o significado do sentinela. Os dias de `pdays` ficam ausentes quando não houve contato anterior; para esses registros, um número imputado não será confundido com o fato histórico porque a flag permanece no conjunto de preditores.

## Divisão treino/teste

`split_data()` usa `stratify=y`. Isso mantém aproximadamente a mesma proporção de `yes` e `no` nos dois subconjuntos. `random_state=42` torna a divisão reproduzível. O teste fica separado e não participa do ajuste do transformador.

In [4]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)
print(f"Treino: {X_train.shape[0]:,} linhas; teste: {X_test.shape[0]:,} linhas")
print("Percentual de cada classe em cada subconjunto:")
class_percentages = pd.DataFrame({
    "treino (%)": y_train.value_counts(normalize=True) * 100,
    "teste (%)": y_test.value_counts(normalize=True) * 100,
}).round(2).sort_index()
display(class_percentages)
print(f"Número de linhas de treino + teste: {len(X_train) + len(X_test):,}")

Treino: 32,950 linhas; teste: 8,238 linhas
Percentual de cada classe em cada subconjunto:


,treino (%),teste (%)
y,,
no,88.73,88.74
yes,11.27,11.26


Número de linhas de treino + teste: 41,188


## Construir o `ColumnTransformer`

A função `make_preprocessor()` identifica as colunas pelo tipo. Nas numéricas, a mediana trata valores ausentes e `StandardScaler` padroniza escalas (útil para a regressão logística). Nas categóricas, a moda trata ausências reais e `OneHotEncoder` converte categorias em indicadores. A categoria textual `unknown` continua sendo uma categoria válida; ela não é substituída pela moda.

O transformador é criado sem ser ajustado. Vamos ajustá-lo somente em `X_train`. Para modelos baseados em árvores, padronizar números não costuma ser necessário; manter um pré-processador compartilhado torna a preparação uniforme, e a escala não altera as divisões por limiar das árvores.

In [5]:
preprocessor = make_preprocessor(X_train)
numeric_columns = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X_train.select_dtypes(exclude=np.number).columns.tolist()
print(f"Colunas numéricas ({len(numeric_columns)}): {numeric_columns}")
print(f"Colunas categóricas ({len(categorical_columns)}): {categorical_columns}")

Colunas numéricas (10): ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'was_previously_contacted']
Colunas categóricas (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


## Ajustar no treino e transformar o teste

`fit_transform(X_train)` aprende medianas, escalas, categorias e frequências usando somente o treino. `transform(X_test)` reaplica os mesmos parâmetros ao teste, sem calcular estatísticas nele. Nas etapas de treinamento e validação cruzada, esse pré-processador será incluído dentro de um `Pipeline` para que cada ajuste seja feito apenas com a parte de treino correspondente.

In [6]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
print(f"Matriz de treino transformada: {X_train_processed.shape}")
print(f"Matriz de teste transformada: {X_test_processed.shape}")
print(f"Quantidade de atributos após One-Hot Encoding: {len(feature_names)}")
print("Primeiros nomes de atributos transformados:")
print(feature_names[:15].tolist())

Matriz de treino transformada: (32950, 63)
Matriz de teste transformada: (8238, 63)
Quantidade de atributos após One-Hot Encoding: 63
Primeiros nomes de atributos transformados:
['numeric__age', 'numeric__campaign', 'numeric__pdays', 'numeric__previous', 'numeric__emp.var.rate', 'numeric__cons.price.idx', 'numeric__cons.conf.idx', 'numeric__euribor3m', 'numeric__nr.employed', 'numeric__was_previously_contacted', 'categorical__job_admin.', 'categorical__job_blue-collar', 'categorical__job_entrepreneur', 'categorical__job_housemaid', 'categorical__job_management']


## Síntese

- `X` não contém `y` nem `duration`.
- O sentinela de `pdays` é separado em uma flag de contato anterior e um valor numérico ausente, que será imputado usando estatísticas aprendidas no treino.
- As classes categóricas são codificadas por One-Hot Encoding, sem perder a categoria `unknown`.
- O teste é estratificado e somente recebe `transform`; nenhum parâmetro é aprendido a partir dele.
- Os objetos `X_train`, `X_test`, `y_train`, `y_test` e `preprocessor` estarão disponíveis para os próximos notebooks. No treinamento, o transformador deve ser colocado dentro de um `Pipeline` do Scikit-learn.

A separação treino/teste avalia generalização; os hiperparâmetros e limiares serão ajustados apenas com validação interna ao treino nas etapas posteriores.